In [ ]:
import os
import pandas as pd
import requests
import shapely
from dotenv import load_dotenv
from io import StringIO
from datetime import timedelta
import geopandas as gpd 
from sklearn.cluster import DBSCAN

load_dotenv()
key = os.getenv("FIRMS_MAP_KEY")
vic_geobox = "140.96,-39.16,149.98,-33.98" #WSEN
aus_geobox = "96,-45,169,-8"
sensors = ["VIIRS_NOAA20_NRT","VIIRS_NOAA21_NRT"]
aus_frames = []
vic_frames = []

for sensor in sensors:
    #get last 3 days of data
    vic_url =f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{key}/{sensor}/{vic_geobox}/3"
    aus_url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{key}/{sensor}/{aus_geobox}/3"
    vic_respond = requests.get(vic_url)
    aus_respond = requests.get(aus_url)

    if vic_respond.status_code == 200 and len(vic_respond.text.splitlines()) > 1:
        temporary1 = pd.read_csv(StringIO(vic_respond.text))
        vic_frames.append(temporary1)

    if aus_respond.status_code == 200 and len(aus_respond.text.splitlines()) > 1:
        temporary2 = pd.read_csv(StringIO(aus_respond.text))
        aus_frames.append(temporary2)

if vic_frames:
    vic_df = pd.concat(vic_frames, ignore_index=True)
    vic_df.to_csv("../data/raw/live/vic_fires.csv", index=False)
if aus_frames:
    aus_df = pd.concat(aus_frames, ignore_index=True)
    aus_df.to_csv("../data/raw/live/aus_fires.csv", index=False)




## Historical Data
Manage historical data logic adding 60-90 days lookup before today's date. Only fetch all once and fetch the gap between last fetch and today on every run to reduce transactions (unless last run is more than 90 days ago).

In [ ]:
temp_df = pd.read_csv(f"https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/{key}/all")
temp_df["min_date"] = pd.to_datetime(temp_df["min_date"]).dt.date
temp_df["max_date"] = pd.to_datetime(temp_df["max_date"]).dt.date
indexed_df = temp_df.set_index("data_id")

earliest = indexed_df.loc[sensors, "min_date"].max()
latest = indexed_df.loc[sensors, "max_date"].min()
lookup_date = latest - timedelta(days=2)

the_path = "../data/raw/historical/vic_fires.csv"
if os.path.exists(the_path) and os.path.getsize(the_path) > 0:
    old = pd.read_csv(the_path)
    if not old.empty:   
        most_recent = pd.to_datetime(old["acq_date"]).dt.date.max()
        gap = max(most_recent + timedelta(days=1), earliest)
    else:
        gap = earliest
else:
    #if there was no historical data fetch from the earliest record
    old = pd.DataFrame()
    gap = earliest



frame = []
if gap < lookup_date:
    for sensor in sensors:
        date_pointer = gap
        while date_pointer < lookup_date:
            total = (lookup_date - date_pointer).days
            range_days = min(5, total) # max fetch only 5 days
            formatted = date_pointer.strftime("%Y-%m-%d")
            url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{key}/{sensor}/{vic_geobox}/{range_days}/{formatted}"
            respond = requests.get(url)
            if respond.status_code == 200 and len(respond.text.splitlines()) > 1:
                temp = pd.read_csv(StringIO(respond.text))
                frame.append(temp)
            date_pointer += timedelta(days = range_days)
            
if frame:
    new = pd.concat(frame, ignore_index=True)
    histo_df = pd.concat([old,new], ignore_index=True)
else:
    histo_df = old

if not histo_df.empty:
    histo_df = histo_df.drop_duplicates(subset=["acq_time", "acq_date", "satellite", "latitude", "longitude"], keep="last")
    histo_df = histo_df[histo_df["acq_date"] >= earliest.strftime("%Y-%m-%d")] #keep the 60-90 days window only
    histo_df.to_csv(the_path, index=False)

In [ ]:
#check the raw live data
df = pd.read_csv("../data/raw/live/vic_fires.csv")
df.info()
df.describe()
df.isna().sum()
df.duplicated(subset=["acq_time", "acq_date", "satellite", "latitude", "longitude"]).sum()
df.sort_values(["bright_ti4"]).head()
df2 = pd.read_csv("../data/raw/live/aus_fires.csv")
df2.sort_values(["bright_ti4"]).head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 305 entries, 0 to 304
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   latitude    305 non-null    float64
 1   longitude   305 non-null    float64
 2   bright_ti4  305 non-null    float64
 3   scan        305 non-null    float64
 4   track       305 non-null    float64
 5   acq_date    305 non-null    object 
 6   acq_time    305 non-null    int64  
 7   satellite   305 non-null    object 
 8   instrument  305 non-null    object 
 9   confidence  305 non-null    object 
 10  version     305 non-null    object 
 11  bright_ti5  305 non-null    float64
 12  frp         305 non-null    float64
 13  daynight    305 non-null    object 
dtypes: float64(7), int64(1), object(6)
memory usage: 33.5+ KB


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
42768,-17.52353,122.89923,207.38,0.57,0.69,2026-11-12,614,N21,VIIRS,l,2.0NRT,306.89,125.82,D
41477,-15.56624,128.23325,207.38,0.52,0.50,2026-11-12,433,N21,VIIRS,l,2.0NRT,312.82,107.42,D
35875,-21.79449,116.34909,207.38,0.36,0.57,2026-11-11,631,N21,VIIRS,l,2.0NRT,308.70,66.15,D
42716,-18.71792,123.43777,207.38,0.58,0.70,2026-11-12,614,N21,VIIRS,l,2.0NRT,311.07,94.93,D
35087,-12.65800,134.89728,207.38,0.39,0.44,2026-11-11,453,N21,VIIRS,l,2.0NRT,316.93,116.06,D


In [ ]:
def data_cleaning(df: pd.DataFrame):
    df = df.copy()
    df["fire_temp"] = df["bright_ti4"]

    df = df.drop_duplicates(subset=["acq_time", "acq_date", "satellite", "latitude", "longitude"], keep="last")
    df = df.dropna(subset=["acq_time", "acq_date", "satellite", "latitude", "longitude","frp", "fire_temp"])
    
    df["confidence"] = df["confidence"].astype(str).str.lower().map({"l":"low", "n":"medium", "h":"high"})
    df["c_level"] = df["confidence"].map({"low": 1, "medium": 2, "high":3})
    
    time = df["acq_time"].astype(str).str.zfill(4)
    dt_utc = pd.to_datetime(df["acq_date"] + " "+ time, format= "%Y-%m-%d %H%M", utc=True)
    df["dt_utc"] = dt_utc
    df["dt_melb"] = dt_utc.dt.tz_convert("Australia/Melbourne")

    df["flagged"] = (df["fire_temp"] < 273) | (df["fire_temp"] > 367)
    df.loc[df["fire_temp"] < 273, "fire_temp"] = 367

    df = df.drop(columns=["scan","track","version", "instrument", "bright_ti4", "bright_ti5"])

    return df

vic_live_cleaned = data_cleaning(pd.read_csv("../data/raw/live/vic_fires.csv"))
aus_live_cleaned = data_cleaning(pd.read_csv("../data/raw/live/aus_fires.csv"))
histo_cleaned = data_cleaning(pd.read_csv("../data/raw/historical/vic_fires.csv"))
vic_live_cleaned.to_csv("../data/processed/live/vic_fires.csv", index=False)
aus_live_cleaned.to_csv("../data/processed/live/aus_fires.csv", index=False)
histo_cleaned.to_csv("../data/processed/historical/vic_fires.csv", index=False)

## Clustering Fire Events

Group detected fire spots that are in adjacent 3 pixels (375m x 3) and within the same day into the same fire event.

In [ ]:
vicdf = pd.read_csv("../data/processed/live/vic_fires.csv")
ausdf = pd.read_csv("../data/processed/live/aus_fires.csv")
histodf = pd.read_csv("../data/processed/historical/vic_fires.csv")

# EPSG VIC: 3111
# EPSG AUS: 3577
# NASA EPSG: 4326
def clustering(df, meter = 1125.0, crs = "EPSG:3577"): 
    df = df.copy()
    df["melb_date"] = pd.to_datetime(df["dt_melb"], utc= True).dt.tz_convert("Australia/Melbourne").dt.date
    gdf = gpd.GeoDataFrame(df, geometry = gpd.points_from_xy(df["longitude"], df["latitude"]), crs = "EPSG:4326").to_crs(crs)
    gdf["event_id"] = -1
    counter = 0 #next available id
    for date, i in gdf.groupby("melb_date").groups.items():
        coor = gdf.loc[i].geometry
        labels = DBSCAN(eps = meter, min_samples = 1).fit_predict(pd.DataFrame({"x": coor.x, "y": coor.y}))
        gdf.loc[i, "event_id"] = labels + counter
        counter += int(labels.max()) + 1

    return pd.DataFrame(gdf.drop(columns="geometry"))

vic_clustered = clustering(vicdf, crs= "EPSG:3111")
aus_clustered = clustering(ausdf)
histo_clustered = clustering(histodf, crs= "EPSG:3111")

def summarise_event(df):
    return df.groupby(["melb_date", "event_id"], as_index=False).agg(
        latitude=("latitude", "mean"), longitude=("longitude", "mean"), 
        avg_temperature=("fire_temp", "mean"), max_temperature = ("fire_temp", "max"),
        avg_intensity =("frp", "mean"), max_intensity=("frp", "max"),
        total_pixels=("frp","count"))

vic_events = summarise_event(vic_clustered)
aus_event = summarise_event(aus_clustered)
histo_event = summarise_event(histo_clustered)
vic_clustered.head()

,latitude,longitude,acq_date,acq_time,satellite,confidence,frp,daynight,fire_temp,c_level,dt_utc,dt_melb,flagged,melb_date,event_id
0,-38.31103,145.80882,2026-11-10,421,N20,medium,3.32,D,340.43,2,2026-11-10 04:21:00+00:00,2026-11-10 15:21:00+11:00,False,2026-11-10,0
1,-37.83333,144.78862,2026-11-10,421,N20,medium,4.27,D,347.33,2,2026-11-10 04:21:00+00:00,2026-11-10 15:21:00+11:00,False,2026-11-10,1
2,-37.79038,143.17043,2026-11-10,423,N20,medium,10.65,D,342.57,2,2026-11-10 04:23:00+00:00,2026-11-10 15:23:00+11:00,False,2026-11-10,2
3,-36.62931,145.04034,2026-11-10,423,N20,medium,20.06,D,354.32,2,2026-11-10 04:23:00+00:00,2026-11-10 15:23:00+11:00,False,2026-11-10,3
4,-36.53992,146.72713,2026-11-10,423,N20,medium,2.27,D,333.10,2,2026-11-10 04:23:00+00:00,2026-11-10 15:23:00+11:00,False,2026-11-10,4


### Example Display

Fire events across Victoria and their clusters over the span of 3 days

In [30]:
import plotly.express as px

fig_pixels = px.scatter_map(
    vic_clustered,
    lat = "latitude",
    lon = "longitude",
    color = vic_clustered["event_id"].astype(str),
    hover_data = ["melb_date", "longitude", "latitude", "frp", "confidence", "daynight", "fire_temp", "flagged"],
    labels = {
        "color": "event ID",
        "longitude": "Longitude",
        "latitude": "Latitude",       
        "frp": "Severity",
        "confidence": "Accuracy",
        "daynight": "Day / Night",
        "fire_temp": "Temperature (K)",
        "flagged": "Overflow",
        "melb_date": "Date (Melb)",
    },
    zoom = 5,
    center = {"lat": -37.0, "lon": 145.0},
    map_style ="carto-darkmatter",
    title ="Detected Fire Pixels",
)
fig_pixels.show()

fig_events = px.scatter_map(
    vic_events,
    lat = "latitude",
    lon = "longitude",
    size = "max_intensity",
    color = vic_events["melb_date"].astype(str),
    hover_data = ["longitude", "latitude", "max_temperature", "avg_temperature", "max_intensity", "avg_intensity", "total_pixels"],
    labels = {
        "color": "Date (Melbourne)",
        "longitude": "Longitude",
        "latitude": "Latitude",  
        "max_intensity": "Max Intensity",
        "avg_intensity": "Avg Intensity",
        "max_temperature": "Max Temp (K)",
        "avg_temperature": "Avg Temp (K)",
        "total_pixels": "Hotspot Pixels",
    },
    zoom=5,
    center = {"lat": -37.0, "lon": 145.0},
    map_style = "carto-darkmatter",
    title = "Clustered Fire Events",
)
fig_events.show()

vic_clustered.head()

,latitude,longitude,acq_date,acq_time,satellite,confidence,frp,daynight,fire_temp,c_level,dt_utc,dt_melb,flagged,melb_date,event_id
0,-38.31103,145.80882,2026-11-10,421,N20,medium,3.32,D,340.43,2,2026-11-10 04:21:00+00:00,2026-11-10 15:21:00+11:00,False,2026-11-10,0
1,-37.83333,144.78862,2026-11-10,421,N20,medium,4.27,D,347.33,2,2026-11-10 04:21:00+00:00,2026-11-10 15:21:00+11:00,False,2026-11-10,1
2,-37.79038,143.17043,2026-11-10,423,N20,medium,10.65,D,342.57,2,2026-11-10 04:23:00+00:00,2026-11-10 15:23:00+11:00,False,2026-11-10,2
3,-36.62931,145.04034,2026-11-10,423,N20,medium,20.06,D,354.32,2,2026-11-10 04:23:00+00:00,2026-11-10 15:23:00+11:00,False,2026-11-10,3
4,-36.53992,146.72713,2026-11-10,423,N20,medium,2.27,D,333.10,2,2026-11-10 04:23:00+00:00,2026-11-10 15:23:00+11:00,False,2026-11-10,4
